# 🔬 Post-Training: SIMPO
Aligns the SFT model using **SIMPO**.

In [ ]:
# ============================================================
# PAPERMILL PARAMETERS (injected at runtime)
# ============================================================
SFT_ADAPTER_PATH = "../01_sft/qwen_medical_arabic_lora"
DATASET_PATH     = "../../data/alignment"
OUTPUT_DIR       = "outputs_simpo"
NUM_EPOCHS       = 2
LEARNING_RATE    = 5e-5
MAX_STEPS        = 10


In [1]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
except Exception:
    pass  # newer unsloth versions don't need this patch

# Load the SFT LoRA model directly — DO NOT call get_peft_model again!
# The model already has LoRA adapters from the SFT stage.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_ADAPTER_PATH,
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_training(model)  # enable gradient checkpointing for training
print("Model loaded for DPO. VRAM:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\Lenovo\anaconda3\envs\unsloth_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3070 Ti Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.4.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded for DPO. VRAM: 2.49 GB


In [2]:
from datasets import load_dataset
import os

import os
    DATA_FILE = os.path.join(DATASET_PATH, "05_simpo_dataset.json")
assert os.path.exists(DATA_FILE), f"Data file not found: {DATA_FILE}"

dataset = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"Dataset loaded: {len(dataset)} examples")

SYSTEM_MSG = "أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية."

def format_pair(example):
    prompt   = f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    chosen   = f"{example['chosen']}<|im_end|>\n"
    rejected = f"{example['rejected']}<|im_end|>\n"
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

dataset = dataset.map(format_pair, remove_columns=dataset.column_names)
print(f"Formatted. Sample count: {len(dataset)}")
print("Sample prompt (first 150 chars):")
print(dataset[0]["prompt"][:150])


Dataset loaded: 300 examples
Formatted. Sample count: 300
Sample prompt (first 150 chars):
<|im_start|>system
أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية.<|im_end|>
<|im_start|>user
أ


In [3]:
from trl import CPOTrainer, CPOConfig

training_args = CPOConfig(
    output_dir          = OUTPUT_DIR,
    max_steps           = MAX_STEPS,
    beta                = 2.0,
    loss_type           = "simpo",
    simpo_gamma         = 0.5,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate       = LEARNING_RATE,
    lr_scheduler_type   = "cosine",
    warmup_ratio        = 0.1,
    max_length          = 1024,
    max_prompt_length   = 512,
    optim               = "adamw_8bit",
    fp16                = not is_bfloat16_supported(),
    bf16                = is_bfloat16_supported(),
    num_train_epochs    = NUM_EPOCHS,
    logging_steps       = 10,
    save_steps          = 50,
    save_total_limit    = 2,
    report_to           = "none",
)

trainer = CPOTrainer(
    model        = model,
    args         = training_args,
    train_dataset= dataset,
    tokenizer    = tokenizer,
)
print("SimPO Trainer ready!")


e:\FineTuning\model_training\02_post_training\unsloth_compiled_cache\UnslothCPOTrainer.py:797: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(
[trl.trainer.cpo_trainer|WARNING]When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.


SimPO Trainer ready!


In [4]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 2 | Total steps = 74
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / rejected,logps / chosen,logits / rejected,logits / chosen,nll_loss
10,3.686000,-3.455526,-2.174159,0.012500,-1.281368,-1.087079,-1.727763,-1.997274,-2.430985,1.727763
20,2.690900,-2.986354,-2.695392,0.312500,-0.290963,-1.347696,-1.493177,-2.094215,-2.433286,1.493177
30,1.948600,-2.763592,-3.609344,0.962500,0.845751,-1.804672,-1.381796,-2.073071,-2.472783,1.381796
40,1.563700,-2.676975,-4.337709,0.986842,1.660735,-2.168855,-1.338488,-2.004717,-2.396070,1.338487
50,1.449900,-2.546409,-4.856416,1.000000,2.310007,-2.428208,-1.273205,-1.942373,-2.322294,1.273205
60,1.370400,-2.494395,-5.182514,1.000000,2.688118,-2.591257,-1.247198,-1.904324,-2.309542,1.247198
70,1.318200,-2.394386,-5.228537,0.987500,2.834151,-2.614268,-1.197193,-1.880134,-2.262192,1.197193


TrainOutput(global_step=74, training_loss=1.9714486598968506, metrics={'train_runtime': 1434.7241, 'train_samples_per_second': 0.418, 'train_steps_per_second': 0.052, 'total_flos': 0.0, 'train_loss': 1.9714486598968506, 'epoch': 1.96})

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved SIMPO model -> qwen_medical_arabic_simpo/")


Saved SIMPO model -> qwen_medical_arabic_simpo/


: 